# Evaluation Results

This notebook loads the completed automatic evaluation metrics and creates four presentation figures using the headline metrics reported in the results table: POPE Adv. F1, Object HalBench CHAIRS, AMBER CHAIR, and AMBER Hal. The two-shot experiment is included where the corresponding metric is available.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

SUMMARY_CSV = REPO_ROOT / 'reports' / 'results_summary.csv'
FIGURE_DIR = REPO_ROOT / 'reports' / 'result_figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_COLORS = {
    'Base LLaVA': '#2F80ED',
    'Direct HSA-DPO': '#F2994A',
    'Standard DPO': '#27AE60',
    'Zero-shot HSA-DPO-M': '#9B51E0',
    'Two-shot HSA-DPO-M': '#C0392B',
}

MODEL_PLOT_LABELS = {
    'Base LLaVA': 'Base',
    'Direct HSA-DPO': 'Direct\nHSA-DPO',
    'Standard DPO': 'Standard\nDPO',
    'Zero-shot HSA-DPO-M': 'Zero-shot\nHSA-DPO-M',
    'Two-shot HSA-DPO-M': 'Two-shot\nHSA-DPO-M',
}

results_df = pd.read_csv(SUMMARY_CSV).rename(columns={
    'pope_f1': 'POPE Adv. F1\n(higher better)',
    'object_chairs': 'Object HalBench CHAIRS\n(lower better)',
    'amber_chair': 'AMBER CHAIR\n(lower better)',
    'amber_hal': 'AMBER Hal\n(lower better)',
})
display(results_df.round(2))
print(f'Figures will be saved to: {FIGURE_DIR.relative_to(REPO_ROOT)}')

## Main Results

- **POPE Adv. F1:** the base model remains slightly best.
- **Object HalBench CHAIRS:** all trained models reduce hallucination compared with the base model; lower is better.
- **AMBER CHAIR / Hal:** trained models reduce hallucination compared with the base model; lower is better.

## Presentation Figures

The following cells save exactly four benchmark figures. Each figure uses only the headline table numbers. Metrics unavailable in an evaluation artifact are omitted from that specific figure, while the other figures keep the available two-shot results.

In [ ]:
def plot_single_metric(metric, title, ylabel, filename, color):
    plot_df = results_df[['model', metric]].copy()
    plot_df['plot_label'] = plot_df['model'].map(MODEL_PLOT_LABELS).fillna(plot_df['model'])
    values = pd.to_numeric(plot_df[metric], errors='coerce')
    plot_df = plot_df[values.notna()].copy()
    values = pd.to_numeric(plot_df[metric], errors='coerce')
    max_value = values.max(skipna=True)

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(
        plot_df['plot_label'],
        values,
        color=[MODEL_COLORS[m] for m in plot_df['model']],
        edgecolor='white',
        linewidth=0.8,
        alpha=0.92,
    )

    labels = [f'{value:.2f}' for value in values]
    ax.bar_label(bars, labels=labels, padding=3, fontsize=9)
    ax.set_title(title, fontsize=15, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Model variant')
    ax.tick_params(axis='x', labelsize=9)
    ax.grid(axis='y', alpha=0.25)
    ax.set_ylim(0, max_value * 1.18 if pd.notna(max_value) and max_value > 0 else 1)
    plt.tight_layout()

    out_path = FIGURE_DIR / filename
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.show()
    return out_path



In [ ]:
saved_figures = [
    plot_single_metric(
        'POPE Adv. F1\n(higher better)',
        'POPE Adversarial Evaluation',
        'F1 score (%) - higher is better',
        'pope_benchmark_histogram.png',
        '#2F80ED',
    ),
    plot_single_metric(
        'Object HalBench CHAIRS\n(lower better)',
        'Object HalBench Evaluation',
        'CHAIRS - lower is better',
        'object_halbench_histogram.png',
        '#F2994A',
    ),
    plot_single_metric(
        'AMBER CHAIR\n(lower better)',
        'AMBER CHAIR Evaluation',
        'Score - lower is better',
        'amber_chair_histogram.png',
        '#27AE60',
    ),
    plot_single_metric(
        'AMBER Hal\n(lower better)',
        'AMBER Hal Evaluation',
        'Score - lower is better',
        'amber_hal_histogram.png',
        '#9B51E0',
    ),
]

display(pd.DataFrame({
    'figure': [p.name for p in saved_figures],
    'path': [str(p.relative_to(REPO_ROOT)) for p in saved_figures],
}))
